In [ ]:
from IPython.display import SVG
import pandas as pd
import seaborn as sb

import json

from parsers.lr.lr0 import LR0Parser
from parsers.lr.slr import SLRParser
from parsers.example_grammars import GRAMMAR_3_20, GRAMMAR_3_23
from parsers.lr.to_graphviz import lr_parser_to_graphviz
from parsers.lr.lr_parsing_engine import LREngine

# Grammar 3.20

This grammar is well described in the book, in cell below, I will build and compare
states graph and final parsing table.

In [ ]:
p = LR0Parser(*GRAMMAR_3_20)

In [ ]:
print(p.to_tabulate())

## Original Parsing Table comparison

Indexes of states differ from the book, but I have to check if more or less,
the functionality is the same. That's why I printed out all states and rules
with their corresponding indexes.

Except the order, whole logic seems to be the same, so my implementation works! 

In [ ]:
p.print_rules_and_states()

In [ ]:
SVG(lr_parser_to_graphviz(p, title="grammar 3.20").create_svg())

## Original Graph comparison

Both look the same, so probably I made a good job!

# Grammar 3.23

This grammar is shown as an example of grammar incompatible with LR(0) parser
generator. It should have duplicated entry in the parsing table, I've already
included this behavior in tests.

## My implementation

In [ ]:
p = LR0Parser(*GRAMMAR_3_23)
print(p.to_tabulate())

In [ ]:
SVG(lr_parser_to_graphviz(p, title="grammar 3.23").create_svg())

Graph is equal to the original one, table also seems to be the same, but I didn't
check it byte-to-byte.

# Grammar 3.23 for SLR parser

Let's check how SLR parser works with grammar 3.23.

In [ ]:
p = SLRParser(*GRAMMAR_3_23)

In [ ]:
print(p.to_tabulate())

* [x] 0
* [x] 1
* [x] 2
* [x] 3
* [x] 4
* [x] 5

Check by hand and compared results to the book table. Looks the same, so 
it's probably working!

In [ ]:
p.print_rules_and_states()

# Parsing in action

Let's run the example on simple test case, let it be grammar 3.23 and SLR parser.

In [ ]:
p = SLRParser(*GRAMMAR_3_23)

In [ ]:
engine = LREngine(p)

In [ ]:
res = engine.parse(["x", "+", "x", "+", "x", "$"])

In [ ]:
print(json.dumps(res, indent=2))

## Exercise 3.8

We have to compare two grammars - one with left recursion, and second with right 
recursion, to see how stack grow differs

### Right recursion 

First we will check the grammar 3.23, because there we already have right
recursion.

In [ ]:
class StackLenDumper:
    def __init__(self):
        self.lens = []

    def dump_stack_len(self, stack, states, symbol, action) -> None:
        self.lens.append(len(stack))

In [ ]:
p = SLRParser(*GRAMMAR_3_23)
engine = LREngine(p)
dumper_right = StackLenDumper()
print(p.to_tabulate())

In [ ]:
for idx, state in p.states.items():
    line = f"{idx:04}\t" + ", ".join(f"[{el}]" for el in state)
    print(line)

In [ ]:
_ = engine.parse(
    ["x", "+", "x", "+", "x", "$"], iteration_callback=dumper_right.dump_stack_len
)

### Left recursion

We will modify a bit grammar 3.23 to get left recursion rather than right one.

The only change is in the second rule, we change `E -> T + E`, into `E -> E + T`.

In [ ]:
modified_grammar_3_23: list[str] = [
    "S -> E $",
    "E -> E + T",
    "E -> T",
    "T -> x",
]

In [ ]:
p = SLRParser(*modified_grammar_3_23)
engine = LREngine(p)
dumper_left = StackLenDumper()
print(p.to_tabulate())

In [ ]:
for idx, state in p.states.items():
    line = f"{idx:04}\t" + ", ".join(f"[{el}]" for el in state)
    print(line)

In [ ]:
_ = engine.parse(
    ["x", "+", "x", "+", "x", "$"], iteration_callback=dumper_left.dump_stack_len
)

### Chart comparison

Let's compare how both stacks behave

In [ ]:
df = pd.DataFrame(
    {"left_recursion": dumper_left.lens, "right_recursion": dumper_right.lens}
)

sb.lineplot(data=df)

### Conclusions

Stack grow higher when we have right recursion. After short analysis of both
parsing tables, I came to conclusions that this is related to the fact, that
when we have left recursion, we reduce more often. After we meet `E`, we are
sure that there is quite a lot to reduce. In comparison, with right recursion,
we put more on the stack delaying the decision.